# Noor Al-Quran — Recitation NLP Training Notebook

This notebook trains a practical Arabic text-matching model for recitation recognition.

## What it does
- Loads the Quran text dataset and the tafsir dataset
- Normalizes Arabic text for better matching
- Builds a TF-IDF character n-gram retrieval model
- Evaluates top-1 and top-3 ayah matching accuracy
- Saves the trained artifacts for reuse in the app

## Dataset sources
- `backend/data/The Quran Dataset.csv`
- `backend/data/Quran with tafsir.csv`

## Model idea
For recitation, the fastest reliable baseline is a retrieval model:
- input: recognized Arabic transcript from the microphone or speech-to-text
- output: the closest matching Quran ayah
- explanation: matched surah, ayah number, and tafsir

This works well for noisy Arabic text and is easy to improve later with a transformer model if needed.

In [3]:
from pathlib import Path
import random
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'backend' / 'data' / 'The Quran Dataset.csv').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve project root containing backend/data/The Quran Dataset.csv')

DATA_DIR = PROJECT_ROOT / 'backend' / 'data'
QURAN_PATH = DATA_DIR / 'The Quran Dataset.csv'
TAFSIR_PATH = DATA_DIR / 'Quran with tafsir.csv'

quran_df = pd.read_csv(QURAN_PATH)
tafsir_df = pd.read_csv(TAFSIR_PATH)

print('Project root:', PROJECT_ROOT)
print('Quran dataset shape:', quran_df.shape)
print('Tafsir dataset shape:', tafsir_df.shape)
print('\nQuran columns:', list(quran_df.columns))
print('\nTafsir columns:', list(tafsir_df.columns))
print('\nQuran sample:')
display(quran_df.head(3))
print('\nTafsir sample:')
display(tafsir_df.head(3))

Project root: /home/abouramt/Desktop/نور-القرآن-(noor-al-quran)
Quran dataset shape: (6236, 19)
Tafsir dataset shape: (6236, 5)

Quran columns: ['surah_no', 'surah_name_en', 'surah_name_ar', 'surah_name_roman', 'ayah_no_surah', 'ayah_no_quran', 'ayah_ar', 'ayah_en', 'ruko_no', 'juz_no', 'manzil_no', 'hizb_quarter', 'total_ayah_surah', 'total_ayah_quran', 'place_of_revelation', 'sajah_ayah', 'sajdah_no', 'no_of_word_ayah', 'list_of_words']

Tafsir columns: ['surah_name_ar', 'surah_name_roman', 'ayah_ar', 'ayah_en', 'tafsir']

Quran sample:


,surah_no,surah_name_en,surah_name_ar,surah_name_roman,ayah_no_surah,ayah_no_quran,ayah_ar,ayah_en,ruko_no,juz_no,manzil_no,hizb_quarter,total_ayah_surah,total_ayah_quran,place_of_revelation,sajah_ayah,sajdah_no,no_of_word_ayah,list_of_words
0,1,The Opener,الفاتحة,Al-Fatihah,1,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"In the Name of Allah—the Most Compassionate, M...",1,1,1,1,7,6236,Meccan,False,NaN,4,"[بِسْمِ,ٱللَّهِ,ٱلرَّحْمَٰنِ,ٱلرَّحِيمِ]"
1,1,The Opener,الفاتحة,Al-Fatihah,2,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ,"All praise is for Allah—Lord of all worlds,",1,1,1,1,7,6236,Meccan,False,NaN,4,"[ٱلْحَمْدُ,لِلَّهِ,رَبِّ,ٱلْعَٰلَمِينَ]"
2,1,The Opener,الفاتحة,Al-Fatihah,3,3,ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"the Most Compassionate, Most Merciful,",1,1,1,1,7,6236,Meccan,False,NaN,2,"[ٱلرَّحْمَٰنِ,ٱلرَّحِيمِ]"



Tafsir sample:


,surah_name_ar,surah_name_roman,ayah_ar,ayah_en,tafsir
0,الفاتحة,Al-Fatihah,بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"In the Name of Allah—the Most Compassionate, M...",باسم الله أبدأ قراءة القرآن، مستعينًا به تعالى...
1,الفاتحة,Al-Fatihah,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ,"All praise is for Allah—Lord of all worlds,",جميع أنواع المحامد من صفات الجلال والكمال هي ل...
2,الفاتحة,Al-Fatihah,ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"the Most Compassionate, Most Merciful,",ثناء على الله تعالى بعد حمده في الآية السابقة.


In [6]:
ARABIC_DIACRITICS_RE = re.compile(r'[\u064B-\u065F\u0670\u06D6-\u06ED]')
ARABIC_PUNCT_RE = re.compile(r"[\u061F\u060C\u061B\.,!\?؛،:\(\)\"'\-ـ]")
SPACE_RE = re.compile(r'\s+')


def normalize_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = ARABIC_DIACRITICS_RE.sub('', text)
    text = text.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
    text = text.replace('ى', 'ي').replace('ة', 'ه')
    text = ARABIC_PUNCT_RE.sub(' ', text)
    text = SPACE_RE.sub(' ', text)
    return text.lower().strip()


quran_norm = quran_df.copy()
quran_norm['surah_name_roman'] = quran_norm['surah_name_roman'].fillna('').astype(str)
quran_norm['ayah_ar_norm'] = quran_norm['ayah_ar'].fillna('').astype(str).map(normalize_arabic)
quran_norm['surah_name_roman_norm'] = quran_norm['surah_name_roman'].fillna('').astype(str).str.lower().str.strip()

# The tafsir file is aligned by surah name + ayah text, so we merge on normalized columns.
tafsir_norm = tafsir_df.copy()
tafsir_norm['surah_name_roman_norm'] = tafsir_norm['surah_name_roman'].fillna('').astype(str).str.lower().str.strip()
tafsir_norm['ayah_ar_norm'] = tafsir_norm['ayah_ar'].fillna('').astype(str).map(normalize_arabic)
tafsir_lookup = (
    tafsir_norm[['surah_name_roman_norm', 'ayah_ar_norm', 'tafsir']]
    .dropna(subset=['tafsir'])
    .drop_duplicates(subset=['surah_name_roman_norm', 'ayah_ar_norm'], keep='first')
)

recitation_df = quran_norm.merge(
    tafsir_lookup,
    on=['surah_name_roman_norm', 'ayah_ar_norm'],
    how='left'
)
recitation_df['tafsir'] = recitation_df['tafsir'].fillna('')
recitation_df['target_id'] = recitation_df['surah_no'].astype(str) + ':' + recitation_df['ayah_no_surah'].astype(str)
recitation_df['target_label'] = recitation_df['surah_name_ar'].astype(str) + ' — ' + recitation_df['ayah_no_surah'].astype(str)
recitation_df['train_text'] = recitation_df['ayah_ar_norm']

matched_tafsir = int(recitation_df['tafsir'].ne('').sum())
print(f'Merged recitation dataset rows: {len(recitation_df)}')
print(f'Tafsir matched rows: {matched_tafsir}')
print('\nMerged sample:')
display(recitation_df[['surah_no', 'surah_name_ar', 'ayah_no_surah', 'ayah_ar', 'tafsir']].head(5))

# Keep a held-out sample for evaluation, but build retrieval index on full corpus.
train_df, test_df = train_test_split(
    recitation_df,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=recitation_df['surah_no']
)

vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=1,
    lowercase=False,
)

X_corpus = vectorizer.fit_transform(recitation_df['train_text'])
retriever = NearestNeighbors(n_neighbors=3, metric='cosine', algorithm='brute')
retriever.fit(X_corpus)

MODEL_DIR = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        'vectorizer': vectorizer,
        'retriever': retriever,
        'corpus': recitation_df[['surah_no', 'surah_name_ar', 'ayah_no_surah', 'ayah_no_quran', 'ayah_ar', 'ayah_en', 'tafsir', 'target_id', 'target_label', 'train_text']].reset_index(drop=True),
        'normalize_version': 'arabic_diacritics_v1',
    },
    MODEL_DIR / 'recitation_retrieval_model.joblib'
)

print(f'Training artifacts saved to: {MODEL_DIR / "recitation_retrieval_model.joblib"}')

Merged recitation dataset rows: 6236
Tafsir matched rows: 6236

Merged sample:


,surah_no,surah_name_ar,ayah_no_surah,ayah_ar,tafsir
0,1,الفاتحة,1,بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,باسم الله أبدأ قراءة القرآن، مستعينًا به تعالى...
1,1,الفاتحة,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ,جميع أنواع المحامد من صفات الجلال والكمال هي ل...
2,1,الفاتحة,3,ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,ثناء على الله تعالى بعد حمده في الآية السابقة.
3,1,الفاتحة,4,مَٰلِكِ يَوْمِ ٱلدِّينِ,تمجيد لله تعالى بأنه المالك لكل ما في يوم القي...
4,1,الفاتحة,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,نخصُّك وحدك بأنواع العبادة والطاعة، فلا نشرك م...


Training artifacts saved to: /home/abouramt/Desktop/نور-القرآن-(noor-al-quran)/backend/models/recitation_retrieval_model.joblib


In [7]:
# -----------------------------
# Evaluate Model Performance
# -----------------------------

corpus_df = recitation_df.reset_index(drop=True)


def inject_recitation_noise(text: str, drop_prob: float = 0.08) -> str:
    # Simulate light ASR errors by dropping some characters/spaces.
    chars = list(text)
    kept = []
    for ch in chars:
        if ch != ' ' and np.random.rand() < drop_prob:
            continue
        kept.append(ch)
    noisy = ''.join(kept)
    noisy = re.sub(r'\s+', ' ', noisy).strip()
    return noisy or text


def predict_top_k(transcript: str, top_k: int = 3):
    cleaned = normalize_arabic(transcript)
    vector = vectorizer.transform([cleaned])
    distances, indices = retriever.kneighbors(vector, n_neighbors=top_k)

    results = []
    for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        row = corpus_df.iloc[idx]
        results.append(
            {
                'rank': rank,
                'distance': float(distance),
                'surah_no': int(row['surah_no']),
                'surah_name_ar': row['surah_name_ar'],
                'ayah_no_surah': int(row['ayah_no_surah']),
                'ayah_ar': row['ayah_ar'],
                'ayah_en': row['ayah_en'],
                'tafsir': row['tafsir'],
                'target_id': row['target_id'],
            }
        )
    return results


def evaluate_retrieval_model(test_frame: pd.DataFrame, top_k: int = 3):
    top1_hits = 0
    topk_hits = 0

    for _, row in test_frame.iterrows():
        noisy_query = inject_recitation_noise(row['train_text'])
        preds = predict_top_k(noisy_query, top_k=top_k)
        pred_ids = [p['target_id'] for p in preds]

        if row['target_id'] == pred_ids[0]:
            top1_hits += 1
        if row['target_id'] in pred_ids:
            topk_hits += 1

    top1_accuracy = top1_hits / len(test_frame)
    topk_accuracy = topk_hits / len(test_frame)
    return top1_accuracy, topk_accuracy


top1_accuracy, top3_accuracy = evaluate_retrieval_model(test_df, top_k=3)
print(f'Top-1 accuracy (with synthetic noise): {top1_accuracy:.4f}')
print(f'Top-3 accuracy (with synthetic noise): {top3_accuracy:.4f}')
print('\nA few sample predictions:')
for sample_text in [
    'بسم الله الرحمن الرحيم',
    'الحمد لله رب العالمين',
    'اهدنا الصراط المستقيم',
    'قل هو الله احد',
]:
    print('\nInput:', sample_text)
    for item in predict_top_k(sample_text, top_k=3):
        print(f"  #{item['rank']} -> {item['surah_name_ar']} | آية {item['ayah_no_surah']} | distance={item['distance']:.4f}")
        print(f"     {item['ayah_ar']}")
        if item['tafsir']:
            print(f"     tafsir: {item['tafsir'][:140]}...")

# -----------------------------
# Save / Reload Example
# -----------------------------
loaded_bundle = joblib.load(MODEL_DIR / 'recitation_retrieval_model.joblib')
print('\nSaved artifact keys:', list(loaded_bundle.keys()))


Top-1 accuracy (with synthetic noise): 0.9519
Top-3 accuracy (with synthetic noise): 0.9818

A few sample predictions:

Input: بسم الله الرحمن الرحيم
  #1 -> الفاتحة | آية 1 | distance=0.4173
     بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
     tafsir: باسم الله أبدأ قراءة القرآن، مستعينًا به تعالى متبركًا بذكر اسمه. وقد تضمنت البسملة ثلاثة من أسماء الله الحسنى، وهي:  «الله» أي: المعبود بحق...
  #2 -> النمل | آية 30 | distance=0.5363
     إِنَّهُۥ مِن سُلَيْمَٰنَ وَإِنَّهُۥ بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
     tafsir: مضمون هذا الكتاب المرسل من سليمان المفتتح بـ«بسم الله الرحمن الرحيم»...
  #3 -> الفاتحة | آية 3 | distance=0.5791
     ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
     tafsir: ثناء على الله تعالى بعد حمده في الآية السابقة....

Input: الحمد لله رب العالمين
  #1 -> الفاتحة | آية 2 | distance=0.4805
     ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ
     tafsir: جميع أنواع المحامد من صفات الجلال والكمال هي له وحده دون من سواه، إذ هو رب كل شيء وخالقه ومدبره. و«العالمون» جمع «عالَم» وهم كل ما سوى الله